In [1]:
import sys
import os
from pathlib import Path

pwd = Path().resolve()
sys.path.append(str(pwd.parent))
sys.path.append(str(pwd / 'src'))

%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import seaborn as sns
from programmable_cubes_UDP import ProgrammableCubes
from programmable_cubes_UDP import programmable_cubes_UDP
import numpy as np
from pygmo import problem
import random
from implementation_heuristic import *
from visual import *
PREFIX = "../"



## Combination of algorithms to achieve best result

In [3]:
PROBLEM = "Enterprise_INV"
udp = programmable_cubes_UDP(PROBLEM,"../")
udp.fitness(np.array([-1]))
ti = udp.initial_cube_types
ci = udp.final_cube_positions
ct = udp.target_cube_positions
tt = udp.target_cube_types
types = np.arange(np.max(ti)+1)



### Show problem

In [ ]:


udp.plot('ensemble',types)
udp.plot('target',types)
debug_plot(udp)


## Branch and bound algorithm

In [ ]:
from heapq import heappop, heappush

OUT_PREFIX = "./chroms/"



DEFAULT_METHODS = [
    lambda udp: find_chromosome_heuristic(udp,budget=1),
    lambda udp: find_chromosome_heuristic(udp,budget=2),
    lambda udp: find_chromosome_heuristic(udp,budget=4),
    lambda udp: find_chromosome_heuristic(udp,budget=2000),
    lambda udp: run_removal_of_stuck_cubes_on_udp(udp,5,False),
    lambda udp: run_removal_of_stuck_cubes_on_udp(udp,5,True),
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=astar_cubes),
    #lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=lambda cubes,id,end,budget: brute_force_move_to(cubes,id,end,budget=budget)),
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=astar_cubes_distance),
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=brute_force_move_to)
]

START_METHODS = [
    lambda udp: find_chromosome_heuristic(udp,budget=1),
    lambda udp: find_chromosome_heuristic(udp,budget=2),
    lambda udp: run_removal_of_stuck_cubes_on_udp(udp,5,True)
]

END_METHODS = [
    #lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=brute_force_move_to),
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=lambda cubes,id,end,budget: brute_force_move_to(cubes,id,end,budget=budget,max_attempts=50,max_moves=30)),
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=lambda cubes,id,end,budget: brute_force_move_to_neighbourhoods(cubes,id,end,budget=budget,max_attempts=50,max_moves=30)),
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=astar_cubes_distance),
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=astar_cubes),
    lambda udp: run_removal_of_stuck_cubes_on_udp(udp,10,True),
    lambda udp: find_chromosome_heuristic(udp,budget=10000),
]

UNSTUCK = [
    lambda udp: run_removal_of_stuck_cubes_on_udp(udp,5,True),
    lambda udp: run_removal_of_stuck_cubes_on_udp(udp,5,True),
    lambda udp: run_removal_of_stuck_cubes_on_udp(udp,5,True)
]


def branch_and_bound(udp:programmable_cubes_UDP,methods=DEFAULT_METHODS,initial_path = ""):
    to_search = [] # contains (fitness, file_path)
    initial_chrom = np.array([])
    initial_fitness = udp.fitness(end_chromosome(initial_chrom))[0]
    ci = np.array(udp.final_cube_positions)
    if initial_path == "":
        initial_path = OUT_PREFIX + "ch.npy"
        np.save(initial_path, initial_chrom)
    else:
        initial_chrom = np.load(initial_path)
        initial_fitness = udp.fitness(end_chromosome(initial_chrom),ci)[0]
    heappush(to_search,(initial_fitness,
                          initial_path
                          ))
    
    
    best = {
        'fitness':initial_fitness,
        'path':initial_path
    }
    
    while len(to_search) > 0:
        prev_fitness, prev_path = heappop(to_search)
        prev_chrom = np.load(prev_path)
        for id,method in enumerate(methods):
            udp.fitness(end_chromosome(prev_chrom),ci)
            new_chrom = method(udp)
            if len(new_chrom) > 0:
                chrom = np.concatenate([prev_chrom,new_chrom])
                new_fitness = udp.fitness(end_chromosome(chrom),ci)[0]
                new_directory = prev_path.removesuffix("ch.npy") + f"{id}/"
                os.makedirs(new_directory,exist_ok = True)
                new_path = new_directory + f"ch.npy"
                np.save(new_path,chrom)

                # Check new best fitness
                if new_fitness < best['fitness']:
                    best['fitness'] = new_fitness
                    best['path'] = new_path
                    print("new best solution", best)
                
                
                if new_fitness < prev_fitness and new_fitness < best['fitness'] + 0.01: # TODO: somehow adaptive criterion, filter when too big
                    # Check if can be improved, 0 means no mistakes which means that it cant be improved
                    wpi, wti = get_first_and_second_mistakes(udp)
                    if len(wpi)+len(wti) == 0:
                        print("Ending prematurely because something achieved 0 mistakes")
                        break # add break to find at least one solution
                    # Put to the search pile
                    heappush(to_search,(new_fitness,
                                        new_path
                                        ))
                    
    
    return best

### Branch and bound from start

In [ ]:

best = branch_and_bound(programmable_cubes_UDP(PROBLEM,PREFIX),DEFAULT_METHODS)

### Branch and bound from checkpioint

In [ ]:
start_path = "enterprise_working_version.npy0/2/0/0/0/2/ch.npy"
start_path = "enterprise_working_version.npy0/2/0/0/0/2/3/0/3/3/0/3/3/0/0/0/ch.npy"
start_path = "enterprise_working_version.npy0/2/0/0/0/2/3/0/3/3/0/3/3/0/0/0/0/3/0/0/3/3/0/0/0/ch.npy"
start_path = "enterprise_working_version.npy0/2/0/0/0/2/3/0/3/3/0/3/3/0/0/0/0/3/0/0/3/3/0/0/0/3/0/0/0/0/3/0/0/0/2/2/4/0/0/3/ch.npy"
start_path = "enterprise_working_version.npy0/2/0/0/0/2/3/0/3/3/0/3/3/0/0/0/0/3/0/0/3/3/0/0/0/3/0/0/0/0/3/0/0/0/2/2/4/4/0/0/2/0/4/ch.npy"
start_path = "enterprise_working_version.npy0/2/0/0/0/2/3/0/3/3/0/3/3/0/0/0/0/3/0/0/3/3/0/0/0/3/0/0/0/0/3/0/0/0/2/2/4/4/0/0/2/0/0/0/ch.npy"
start_path = "enterprise_24_mistakes.npy"
start_path = "enterprise_24_mistakes.npy4/0/0/0/ch.npy"
start_path = "enterprise_19_mistakes.npy"
start_path = "enterprise_19_mistakes.npy4/5/1/5/0/0/ch.npy"
start_path = "enterprise_11_mistakes.npy"
start_path = "enterprise_12_mistakes_no_second_mistakes.npy"
start_path = "enterprise_11_mistakes_no_second_mistakes.npy"
start_path = "enterprise_9_mistakes_no_second_mistakes.npy"
start_path = "enterprise_8_mistakes_no_second_mistakes.npy"
start_path = "enterprise_2_mistakes_no_second_mistakes.npy"
start_path = "enterprise_2_mistakes_no_second_mistakes_so_close.npy"
start_path = "enterprise_1_mistakes_no_second_mistakes.npy"
start_path = "enterprise_1_mistakes_no_second_mistakes_another.npy"

#start_path = OUT_PREFIX + "ch.npy"

In [ ]:
TEST = [
    lambda udp: find_chromosome(udp,pairing=pair_colours,pathfinding=lambda cubes,id,end,budget: 
                                brute_force_move_to_neighbourhoods(cubes,id,end,budget=budget,max_attempts=100,max_moves=20),verbose=True),
]
for i in range(20):
    result = branch_and_bound(programmable_cubes_UDP(PROBLEM,PREFIX),END_METHODS,start_path)
    print(result["path"])
    start_path = result["path"]

#### Optional: create initial condition

In [ ]:
print(udp.fitness(np.array([-1])))
ci = udp.final_cube_positions
chrom = []
print(udp.fitness(end_chromosome(chrom),ci),analyze_first_and_second_mistakes(udp))



In [ ]:
chrom = np.concatenate([chrom,find_chromosome_heuristic(udp,budget=100)])
print(udp.fitness(end_chromosome(chrom),ci),analyze_first_and_second_mistakes(udp))

In [ ]:
# Save it where an optimization would start
np.save(OUT_PREFIX + "ch.npy",chrom)

## Manual end

### Load

In [ ]:

chrom = np.load(start_path)
chrom = np.array(chrom,dtype=int)
filtered_chrom = filter_impossible_moves(udp,chrom)





In [ ]:
print(udp.fitness(end_chromosome(chrom),ci),analyze_first_and_second_mistakes(udp))

### Stuck removal

In [ ]:
chrom = np.concatenate([chrom,run_removal_of_stuck_cubes_on_udp(udp,5,True,pathfinding=astar_cubes,verbose=True)])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_colours,pathfinding=lambda cubes,id,end,budget: astar_with_bridge(cubes,id,end,budget=budget))])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

### Brute force to transfer stuckness onto other and possibly solve it

In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_colours,pathfinding=lambda cubes,id,end,budget: 
                                brute_force_move_to(cubes,id,end,budget=budget,max_attempts=10000,max_moves=50))])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))


In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_shuffled,pathfinding=astar_cubes,move_only_freeroaming_cubes_at_target_position=True)])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_colours,
                                              pathfinding=lambda cubes,id,end,budget: brute_force_move_to(cubes,id,end,budget=budget,max_attempts=50000,max_moves=50))])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_colours,pathfinding=astar_cubes_distance)])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_colours,pathfinding=astar_cubes)])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_colours,pathfinding=force_random_move_pathfinding)])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

### Match to closest

In [ ]:
chrom = np.concatenate([chrom,find_chromosome_heuristic(udp,budget=20000)])
print(udp.fitness(end_chromosome(chrom)),analyze_first_and_second_mistakes(udp))

## Save and export to analyze configuration

In [ ]:
np.save("enterprise_solved",chrom)

save_achieved_config(udp,chrom)
save_mistakes(udp)